# PAN 2026 – Generated Plagiarism Detection
## Exploratory Data Analysis

This notebook walks through:
1. Loading the training corpus
2. Inspecting label distribution and text statistics
3. Comparing stylometric features between human and AI-generated texts
4. Evaluating the zero-shot perplexity baseline

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Training Data
Adjust the path below to point to your downloaded training corpus.

In [ ]:
from src.data_loader import load_jsonl, to_dataframe

TRAIN_PATH = '../data/raw/train.jsonl'
records = load_jsonl(TRAIN_PATH)
df = to_dataframe(records)
print(f'Loaded {len(df)} documents')
df.head()

## 2. Label Distribution

In [ ]:
label_counts = df['label'].value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(5, 3))
label_counts.rename({0: 'Human', 1: 'AI-generated'}).plot.bar(ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Label distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Basic Text Statistics

In [ ]:
df['char_len'] = df['text'].str.len()
df['word_len'] = df['text'].str.split().str.len()

df.groupby('label')[['char_len', 'word_len']].describe().round(1)

## 4. Stylometric Features

In [ ]:
from src.features.stylometric import StylometricFeatures

sty = StylometricFeatures()
sty_feats = sty.extract_batch(df['text'].tolist())
sty_df = pd.DataFrame(sty_feats)
sty_df['label'] = df['label'].values
sty_df.head()

In [ ]:
feat_cols = [c for c in sty_df.columns if c != 'label']
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for ax, col in zip(axes, feat_cols):
    for lbl, colour in [(0, 'steelblue'), (1, 'tomato')]:
        data = sty_df.loc[sty_df['label'] == lbl, col].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=colour, label=['Human', 'AI'][lbl])
    ax.set_title(col)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 5. Zero-Shot Perplexity Baseline
> ⚠️ This cell downloads GPT-2 and DistilGPT-2 weights (~500 MB) and may take several minutes.

In [ ]:
from src.models.zero_shot import PerplexityThresholdDetector
from src.evaluate import compute_metrics, print_report

# Use a subset for quick exploration
sample = df.sample(min(200, len(df)), random_state=42)
texts = sample['text'].tolist()
labels = sample['label'].tolist()

detector = PerplexityThresholdDetector(threshold=50.0)
preds = detector.predict(texts)
y_pred = [p['label'] for p in preds]
y_score = [p['score'] for p in preds]

print_report(labels, y_pred, y_score, title='Zero-shot Perplexity Baseline (sample)')

## 6. Threshold Sweep

In [ ]:
from src.evaluate import optimal_threshold

best_thresh, best_f1 = optimal_threshold(labels, y_score, metric='f1_macro')
print(f'Best threshold: {best_thresh:.3f}  (F1-macro={best_f1:.4f})')

## 7. Embedding Self-Similarity

In [ ]:
from src.features.embeddings import EmbeddingFeatures

emb = EmbeddingFeatures()
emb_feats = emb.extract_batch(texts)
self_sims = [f['self_sim'] for f in emb_feats]

human_sims = [s for s, l in zip(self_sims, labels) if l == 0 and not np.isnan(s)]
ai_sims    = [s for s, l in zip(self_sims, labels) if l == 1 and not np.isnan(s)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(human_sims, bins=30, alpha=0.6, label='Human', color='steelblue')
ax.hist(ai_sims,    bins=30, alpha=0.6, label='AI-generated', color='tomato')
ax.set_xlabel('Intra-document sentence self-similarity')
ax.set_title('Self-similarity distribution')
ax.legend()
plt.tight_layout()
plt.show()